In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch5. LSTM(Long Short-Term Memory ; RNN)으로 영화평 구분하기</font>**
- imdb의 5만개 영화 감상평(독립변수) - 부정/긍정(종속변수)

# 1. 패키지 import

In [2]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from time import time # 70.1.1부터 현재까지 몇초가 지났는지

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense#, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score
import pandas as pd

# 2. 하이퍼 파라미터 설정(이 파라미터를 바꾸면 모델 score나 학습속도에 차이)

In [83]:
MY_WORDS = 10000 # imdb 데이터안의 단어 수
MY_LENGTH = 80 # 영화평 단어수 80개까지만 독립변수 (200추천)
MY_EMBED = 32  # 임베딩 layer의 출력 차원(256추천)
MY_HIDDEN = 64 # LSTM의 units수(128추천)

MY_EPOCH = 10 # 반복 학습 수(fit 20추천)
MY_BATCH = 200 # 매번 가져오는 데이터수 
# 불용어 인덱스 설정(빈도수 상위 40개는 제외 : the, a, is등)
SKIP_TOP = 40

# 3. 데이터 불러오기

In [4]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=MY_WORDS) # MY_WORDS(10000)개

17464789/17464789 [==============================] - 1s 0us/step


In [11]:
print('학습용 데이터 shape :', X_train.shape, y_train.shape)
print('학습용 입력 데이터 샘플 :', X_train[0], '-', type(X_train[0]), '-', len(X_train[0]))
print('학습용 타겟 데이터 샘플(0:부정/1:긍정) :', y_train[0])

print('시험용 데이터 shape :', X_test.shape, y_test.shape)
print('시험용 입력 데이터 샘플 :', X_test[0], '-', type(X_test[0]), '-', len(X_test[0]))
print('시험용 타겟 데이터 샘플(0:부정/1:긍정) :', y_test[0])

학습용 데이터 shape : (25000,) (25000,)
학습용 입력 데이터 샘플 : [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32] - <class '

In [18]:
# X_train의 길이 및 평균길이
print([len(x) for x in X_train[:10]])
np.array([len(x) for x in X_train]).mean()

[218, 189, 141, 550, 147, 43, 123, 562, 233, 130]


238.71364

In [20]:
# 부/긍정 갯수 확인
print('학습용 데이터의 긍정 갯수 :', y_train.sum())
print('테스트용 데이터의 긍정 갯수 :', y_test.sum())

학습용 데이터의 긍정 갯수 : 12500
테스트용 데이터의 긍정 갯수 : 12500


In [24]:
# 부/긍정 갯수 확인
pd.Series(y_train).value_counts()#.sort_index()

1    12500
0    12500
dtype: int64

# 4. 문자 단어 -> 정수

In [32]:
word_to_id = imdb.get_word_index() # dict (단어:id값) : 빈도수가 높은 단어를 id앞에
print(word_to_id['movie'])
print(word_to_id['film'])
print(word_to_id['the'])
# 정수 : 단어 dict
id_to_word = {}
for word, id in word_to_id.items():
    id_to_word[id] = word
print(id_to_word[17])
print(id_to_word[19])
print(id_to_word[1])

17
19
1
movie
film
the


In [34]:
# 감성분성에서 필요없는 조사, 관사
print([id_to_word.get(i) for i in range(1, 40)])

['the', 'and', 'a', 'of', 'to', 'is', 'br', 'in', 'it', 'i', 'this', 'that', 'was', 'as', 'for', 'with', 'movie', 'but', 'film', 'on', 'not', 'you', 'are', 'his', 'have', 'he', 'be', 'one', 'all', 'at', 'by', 'an', 'they', 'who', 'so', 'from', 'like', 'her', 'or']


In [52]:
msg = 'What a wonderful movie'
msg = msg.lower().split()
# 1: 리뷰시작할때 무조건 추가, 2:1000개문자가 짤려서 잘못 읽어옴, 3:padding처리
data = [1] + [word_to_id.get(m, -1)+3 for m in msg]
print('원 후기 내용 :', msg)
print('encoding된 data :', data)
data_msg = ' '.join([id_to_word.get(d-3, '??') for d in data])
print('data로 추정된 msg :', data_msg)

원 후기 내용 : ['what', 'a', 'wonderful', 'movie']
encoding된 data : [1, 51, 6, 389, 20]
data로 추정된 msg : ?? what a wonderful movie


# 5. 숫자 영화평 -> 자연어 영화평 함수

In [64]:
def decoding(review_num):
    "숫자 영화평을 자연어 영화평으로 바꿔 return"
    decoded = [id_to_word.get(d-3, '??') for d in review_num]
    return ' '.join(decoded)
decoding(X_train[0])

"?? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ?? is an amazing actor and now the same being director ?? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ?? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ?? to the two little boy's that played the ?? of norman and paul they were just brilliant children are often left out of the ?? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't

# 6. 영화평 데이터 처음 5개 길이 출력 함수

In [67]:
def show_length(X_train=X_train):
    print('첫 데이터 5개 영화평 단어 길이')
    for i in range(5):
        print(f'{i}번째 길이 : {len(X_train[i])}')

In [68]:
print('pad sequence 작업 전')
show_length(X_train)
show_length(X_test)

pad sequence 작업 전
첫 데이터 5개 영화평 단어 길이
0번째 길이 : 218
1번째 길이 : 189
2번째 길이 : 141
3번째 길이 : 550
4번째 길이 : 147
첫 데이터 5개 영화평 단어 길이
0번째 길이 : 68
1번째 길이 : 260
2번째 길이 : 603
3번째 길이 : 181
4번째 길이 : 108


In [73]:
# 영화평 단어길이 최대값과 최소값
max(len(x) for x in X_train), min(len(x) for x in X_train)

(2494, 11)

# 7. 모든 영화평의 길이를 동일하게(MY_LENGTH만큼)

In [74]:
MY_LENGTH

80

In [76]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((25000,), (25000,), (25000,), (25000,))

In [77]:
X_train = pad_sequences(X_train,
                       maxlen=MY_LENGTH,
                       truncating='post', # 80단어 이상일 경우 어디를 짜를지 여부
                       # truncating='pre',
                       padding='pre', # 80단어 미만일 경우 앞에 zero를 붙임
                       #padding='post' 
                       )
X_test = pad_sequences(X_test,
                       maxlen=MY_LENGTH,
                       truncating='post', # 80단어 이상일 경우 어디를 짜를지 여부
                       # truncating='pre',
                       padding='pre', # 80단어 미만일 경우 앞에 zero를 붙임
                       #padding='post' 
                       )
show_length(X_train)
show_length(X_test)

첫 데이터 5개 영화평 단어 길이
0번째 길이 : 80
1번째 길이 : 80
2번째 길이 : 80
3번째 길이 : 80
4번째 길이 : 80
첫 데이터 5개 영화평 단어 길이
0번째 길이 : 80
1번째 길이 : 80
2번째 길이 : 80
3번째 길이 : 80
4번째 길이 : 80


In [82]:
X_test[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    1,  591,  202,   14,   31,    6,  717,   10,   10,    2,
          2,    5,    4,  360,    7,    4,  177, 5760,  394,  354,    4,
        123,    9, 1035, 1035, 1035,   10,   10,   13,   92,  124,   89,
        488, 7944,  100,   28, 1668,   14,   31,   23,   27, 7479,   29,
        220,  468,    8,  124,   14,  286,  170,    8,  157,   46,    5,
         27,  239,   16,  179,    2,   38,   32,   25, 7944,  451,  202,
         14,    6,  717])

In [81]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((25000, 80), (25000,), (25000, 80), (25000,))

# 9. 모델 생성 구현

In [88]:
model = Sequential(name='sequential')
model.add(Embedding(input_dim=MY_WORDS, # 임베딩 입력 : 10000
                    output_dim=MY_EMBED, # 임베딩 출력 : 32
                    input_length=MY_LENGTH)) # 입력 단어 수 : 80
# RNN은 장기 과거 시점의 모델 학습이 어려움. 길이가 길어질수록 과거 학습에 어려움이 발생.
# RNN 개선모델 1.LSTM 2.GRU
model.add(LSTM(units=MY_HIDDEN, # 출력 units
          dropout=0.3, # 0.2~0.5 각 스텝마다 LSTM셀로 들어가는 X 연결 30%를 0으로
          recurrent_dropout=0.2, # GPU안 씀. 0.1~0.3 이전 LSTM에서 다음 LSTM으로 가는 c연결 20%를 0으로
          kernel_regularizer=l2(0.001), # 입력 가중치 L2규제
          recurrent_regularizer=l2(0.001), # 순환가중치 L2규제
          kernel_initializer='he_normal', # 입력 가중치 초기화를 he방법으로
          recurrent_initializer='orthogonal'))  # 순환가중치 초기화
model.add(Dense(units=1, activation='sigmoid'))
model.summary()
# embedding 층 : 파라미터 입력*출력 = 10000*32 
# LSTM 층 : 입력*출력 + (64*64) + unit수(bias) = [(32*64) + (64*64) + 64] * 4(LSTM은 게이트가 4)
#                                           =  (2048+4096+64)*4
# Dense층 : 입력*출력 + 출력 = 64*1+1 = 65

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 80, 32)            320000    
                                                                 
 lstm_2 (LSTM)               (None, 64)                24832     
                                                                 
 dense_2 (Dense)             (None, 1)                 65        
                                                                 
Total params: 344,897
Trainable params: 344,897
Non-trainable params: 0
_________________________________________________________________


# 10. 학습설정 및 학습하기

In [89]:
model.compile(loss='binary_crossentropy',
             optimizer='adam',
             metrics=['acc'])
begin = time() # 70.1.1부터 현재까지의 초수
earlyStopping = EarlyStopping(monitor='val_loss',
                     patience=3,
                     mode='min',
                     restore_best_weights=True, # 멈췄을 경우, val_loss가 가장 낮았던 시점의 가중치로 복원
                     verbose=1) # 언제 멈췄는지 로그 출력
hist = model.fit(X_train, y_train,
                validation_split=0.2,
                epochs=MY_EPOCH,
                batch_size=MY_BATCH,
                verbose=2,
                callbacks=[earlyStopping])
end = time()
print(f'총 학습 시간 : {end-begin:.2f}초')

Epoch 1/10
100/100 - 22s - loss: 1.0140 - acc: 0.6568 - val_loss: 0.7401 - val_acc: 0.8010 - 22s/epoch - 221ms/step
Epoch 2/10
100/100 - 20s - loss: 0.6147 - acc: 0.8314 - val_loss: 0.6289 - val_acc: 0.7986 - 20s/epoch - 201ms/step
Epoch 3/10
100/100 - 20s - loss: 0.4412 - acc: 0.8787 - val_loss: 0.5260 - val_acc: 0.8136 - 20s/epoch - 199ms/step
Epoch 4/10
100/100 - 20s - loss: 0.3461 - acc: 0.8990 - val_loss: 0.4911 - val_acc: 0.8102 - 20s/epoch - 203ms/step
Epoch 5/10
100/100 - 21s - loss: 0.2913 - acc: 0.9094 - val_loss: 0.5653 - val_acc: 0.8054 - 21s/epoch - 209ms/step
Epoch 6/10
100/100 - 20s - loss: 0.2503 - acc: 0.9221 - val_loss: 0.5591 - val_acc: 0.8042 - 20s/epoch - 201ms/step
Epoch 7/10
Restoring model weights from the end of the best epoch: 4.
100/100 - 20s - loss: 0.2281 - acc: 0.9280 - val_loss: 0.5198 - val_acc: 0.7966 - 20s/epoch - 200ms/step
Epoch 7: early stopping
총 학습 시간 : 143.48초
